<!-- nav -->
[← 7 · JSON for machines, YAML for people](07_JSON_And_YAML.ipynb) · [Home](../../README.md) · [9 · When requirements change: evolving schemas →](09_When_Requirements_Change.ipynb)

# 8 · Tools that work for every schema

**The problem.** Your team keeps writing the same utilities for each record type: an audit log that says what
changed, a data dictionary for the wiki, a debug dump. With classes, each utility either knows every class or leans
on reflection tricks that break on the next refactor.

Here, two things are data you can walk:

1. **Schemas.** A schema is an ordinary value. A tool can read its properties, adjacencies, relations and `unique`
   clauses and generate documentation, forms or code from them.
2. **Instances.** Every instance is *Visitable*. `obj.accept(visitor)` writes the object's properties and entries into
   any visitor you hand it. The object doesn't know what the visitor does with them: a serializer saves them, the
   validator checks them, and your tool can do anything else.

This case study builds both kinds of tool, and they work unchanged on every schema from the earlier case studies.

In [1]:
from collections import defaultdict

from mbse.Schemas.Framework import Proxies, Reachable, Schemas

store = Proxies.OfStore()

# The address book from case study 2, compactly.
Contact = Schemas.OfObject.Builder().name('Contact').ref().properties(
    lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
    lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    lambda prop: prop.name('vip').of(lambda t: t.as_native(bool)),
).create()
Address = Schemas.OfObject.Builder().name('Address').ref().properties(
    lambda prop: prop.name('street').of(lambda t: t.as_native(str)),
    lambda prop: prop.name('country_code').of(lambda t: t.as_native(str)),
).create()
ContactAddresses = (
    Schemas.OfRelation.Builder().name('ContactAddresses')
    .links('contact', 'address')
    .properties(lambda prop: prop.name('label').of(lambda t: t.as_native(str)))
    .unique('address', 'label')
    .create()
)
Contact = Schemas.OfObject.Builder(Contact).relations(lambda adj: adj.name('addresses').of(ContactAddresses).me('contact')).update()
Address = Schemas.OfObject.Builder(Address).relations(lambda adj: adj.name('residents').of(ContactAddresses).me('address')).update()
for schema in [Contact, Address, ContactAddresses]:
    store.register(schema)

## Tool 1: a data dictionary, generated from schemas

`Schemas.OfObject.Data` has `properties` (name to type schema) and `adjacencies` (name to adjacency). A relation has
`links`, `properties` and `uniques`. `store.name_of(schema)` gives the registered name. That's all a
documentation generator needs:

In [2]:
def type_name(schema):
    if isinstance(schema, Schemas.OfNative.Data):
        return schema.type.__name__
    return type(schema).__name__


def data_dictionary(*schemas):
    lines = []
    for schema in schemas:
        lines += [f'### {store.name_of(schema)}', '', '| Name | Kind | Details |', '|---|---|---|']
        for name, prop in schema.properties.items():
            lines.append(f'| `{name}` | property | {type_name(prop)} |')
        for name, adjacency in schema.adjacencies.items():
            relation = adjacency.relation
            others = [link for link in relation.links if link != adjacency.me]
            keys = '; '.join(f"unique({', '.join(sorted(u))})" for u in relation.uniques) or 'no cardinality limit'
            lines.append(f'| `{name}` | relation `{store.name_of(relation)}` as `{adjacency.me}` '
                         f'| links {", ".join(others)}; properties {", ".join(relation.properties) or "none"}; {keys} |')
        lines.append('')
    return '\n'.join(lines)


from IPython.display import Markdown

Markdown(data_dictionary(Contact, Address))

### Contact

| Name | Kind | Details |
|---|---|---|
| `given_name` | property | str |
| `family_name` | property | str |
| `vip` | property | bool |
| `addresses` | relation `ContactAddresses` as `contact` | links address; properties label; unique(address, label) |

### Address

| Name | Kind | Details |
|---|---|---|
| `street` | property | str |
| `country_code` | property | str |
| `residents` | relation `ContactAddresses` as `address` | links contact; properties label; unique(address, label) |


The same walk could emit a JSON Schema, a TypeScript interface or an HTML form. This is the "schemas are values"
decision from case study 1 paying off, and it's how generated bindings for other languages will be produced.

## Tool 2: reading any object, with a visitor

A visitor is a set of small handles. `accept` calls them in a fixed pattern. For each property it calls:

```python
visitor.property(name, lambda p: p.value(lambda v: v.as_native(lambda n: n.set(value))))
```

and for each entry of each adjacency:

```python
visitor.adjacency(name, lambda a: a.add(lambda e: (e.link(other, lambda k: k.set(target)),
                                                   e.property(name, lambda p: ...))))
```

That's the object *writing itself* into the visitor, handle by handle: a property handle `p`, the value handle `v`
which dispatches on kind (`as_native`), the native slot `n`, an adjacency `a`, an entry `e`, and a link `k`. Each
handle passes the next one to a callback instead of returning it. That's why the same object can be written into
a serializer, a validator or a builder without allocating anything in between.

To read any object into a dict, implement those handles so that `set` stores what it's given:

In [3]:
class Slot:
    """Handles for one value: `p.value(...)` -> `v.as_native(...)` -> `n.set(value)` store into `target[name]`."""

    def __init__(self, target, name):
        self.target, self.name_ = target, name

    def value(self, callback):     # Visitors.OfProperty.value
        callback(self)
        return self

    def as_native(self, callback):  # Visitors.OfAny.as_native: this visitor handles native values only
        callback(self)
        return self

    def set(self, value):          # Visitors.OfNative.set
        self.target[self.name_] = value
        return self


class Entry:
    """Handles for one entry: links and properties land in one row."""

    def __init__(self, row):
        self.row = row

    def link(self, name, callback):   # Visitors.OfEntry.link -> Visitors.OfLink.set
        callback(Slot(self.row, name))
        return self

    def property(self, name, callback):
        callback(Slot(self.row, name))
        return self


class Record:
    """A write-only Visitors.OfObject: collects what an object writes into it."""

    def __init__(self):
        self.properties, self.adjacencies = {}, defaultdict(list)

    def property(self, name, callback):
        callback(Slot(self.properties, name))
        return self

    def adjacency(self, name, callback):
        rows = self.adjacencies[name]

        class Adjacency:
            def add(self, fill):
                row = {}
                fill(Entry(row))
                rows.append(row)
                return self

        callback(Adjacency())
        return self


def record(obj):
    visitor = Record()
    obj.accept(visitor)
    return visitor

That's a complete reader in about 50 lines, and it doesn't mention `Contact`, `Address` or any property name. (A
full visitor, like the builders and serializers, implements the whole `Visitors.OfObject` protocol so it can be
*read from* as well. This one only needs to be *written into*, so it implements just what `accept` calls.)

In [4]:
home = store.Address().street('7 Old Road').country_code('GB').create()
alice = (
    store.Contact()
    .given_name('Alice').family_name('Liddell').vip(False)
    .addresses(lambda x: x.address(home).label('home'))
    .create()
)

r = record(alice)
print(r.properties)
print({name: [{k: getattr(v, 'street', v) for k, v in row.items()} for row in rows] for name, rows in r.adjacencies.items()})

{'given_name': 'Alice', 'family_name': 'Liddell', 'vip': False}
{'addresses': [{'address': '7 Old Road', 'label': 'home'}]}


## Tool 3: an audit log, for any schema

With a generic reader, "what changed?" is a dictionary comparison. Linked objects are compared by identity, the same
equality the framework uses for entries:

In [5]:
def snapshot(obj):
    r = record(obj)
    entries = {
        (name, tuple(sorted((k, id(v) if hasattr(v, 'accept') else v) for k, v in row.items())))
        for name, rows in r.adjacencies.items()
        for row in rows
    }
    return r.properties, entries


def changes(before, after):
    (props_a, entries_a), (props_b, entries_b) = before, after
    for name in sorted(props_a.keys() | props_b.keys()):
        if props_a.get(name, '<absent>') != props_b.get(name, '<absent>'):
            print(f'  {name}: {props_a.get(name, "<absent>")!r} -> {props_b.get(name, "<absent>")!r}')
    for adjacency, entry in sorted(entries_b - entries_a):
        print(f'  + {adjacency} {dict(entry)}')
    for adjacency, entry in sorted(entries_a - entries_b):
        print(f'  - {adjacency} {dict(entry)}')


before = snapshot(alice)
(
    store.Contact(alice)
    .family_name('Hargreaves')
    .vip(lambda v: v.clear())
    .addresses(lambda x: x.address(lambda a: a.street('2 New Lane').country_code('GB')).label('home'))
    .update()
)
changes(before, snapshot(alice))

  family_name: 'Liddell' -> 'Hargreaves'
  vip: False -> '<absent>'
  + addresses {'address': 4360590384, 'label': 'home'}


(The `+` entry shows the new address by object id. A real audit log would label objects with a symbol, as snapshots
do. Whether the framework should offer such labels to tools is an open question in `FRAMEWORK.md`.)

## Tool 4: dump a whole component

Combine the reader with `Reachable.of` from case study 4 to dump everything connected to an object, whatever its
schema:

In [6]:
for obj in Reachable.of(home):
    print(f'{obj.schema_name():8}', record(obj).properties)

Address  {'street': '7 Old Road', 'country_code': 'GB'}
Contact  {'given_name': 'Alice', 'family_name': 'Hargreaves'}
Address  {'street': '2 New Lane', 'country_code': 'GB'}


## Why visitors, and not `obj.to_dict()`?

- **One protocol, many tools.** Serializers, validators, reachability and your tools all consume the same calls.
  A new implementation of objects (generated classes, say) only needs `accept`, and every tool works with it.
- **Both directions.** The same handles are used to *write* data. A builder is a visitor too: `FromPlain` loads a
  snapshot by making calls like the ones `accept` makes, into a builder.
- **No intermediate structures.** Values flow straight from the object into the tool, and a validator never builds
  a copy of your graph.
- **Schema-agnostic.** Nothing above imports a schema. The tools work on data they've never seen.

## What you learned

- Schemas are data. Walk `properties`, `adjacencies`, `links` and `uniques` to generate documentation or code.
- `obj.accept(visitor)` writes an object into any visitor, through handles passed to callbacks.
- A small write-only visitor reads any object. Diffing, dumping and auditing follow from it.

**Next:** requirements change. Case study 9 evolves schemas (new fields, versions, variants) and marks where the
framework's design is still open.

<!-- nav -->
[← 7 · JSON for machines, YAML for people](07_JSON_And_YAML.ipynb) · [Home](../../README.md) · [9 · When requirements change: evolving schemas →](09_When_Requirements_Change.ipynb)